# ISA 414 - Homework 6
### FAA flights: migrating to MongoDB and querying

In [ ]:
%pip install pymongo --upgrade

#### Task 1: load the original data

In [ ]:
import pandas as pd

df = pd.read_csv("flights.csv")
df

#### Connecting to MongoDB

In [ ]:
import pymongo

# connecting to MongoDB
# replace <db_password> with your database user password (remove the < > symbols)

# Mac version (from Lecture 11), avoids permission/SSL errors
%pip install certifi
import certifi
connection_string = "mongodb+srv://patricrw_db_user:<YOUR_PASSWORD>@cluster0.t2bfd6o.mongodb.net/?appName=Cluster0"
client = pymongo.MongoClient(connection_string, tlsCAFile=certifi.where())

#### Task 2: create a new collection called *flights* inside the *ISA414* database

In [ ]:
db = client.ISA414
collection = db.flights

#### Task 3: populate the flights collection

In [ ]:
# the argument "records" creates a list like [{column -> value}, ... , {column -> value}]
data_json = df.to_dict("records")

# inserting the data
collection.insert_many(data_json)

#### Query 1: return only the carrier codes for flights delayed at departure (dep_delay > 0)

In [ ]:
result = collection.find({"dep_delay":{"$gt":0}}, {"carrier":1, "_id":0})

for document in result:
    print(document)

#### Query 2: fix the carrier code of American (the right code is AA)

In [ ]:
query      = {"carrier":"American"}
new_values = {"$set":{"carrier":"AA"}}

collection.update_many(query, new_values)

# print the American flights after the update
for result in collection.find({"carrier":"AA"}):
    print(result)

#### Query 3: calculate an estimate of the average delay at departure per carrier

In [ ]:
result = collection.aggregate([
    {"$match": {"dep_delay":{"$gt":-1000}}},
    {"$group": {"_id":"$carrier", "avg_dep_delay":{"$avg":"$dep_delay"}}}
])

for document in result:
    print(document)